In [1]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"

response = requests.get(url)
response.encoding = "utf-8"

print(response.status_code)

200


##### 200 means roughly: Server received my request and successfully returned the page.

In [2]:
soup = BeautifulSoup(response.text, "html.parser")

books = soup.find_all("article", class_="product_pod")

print(len(books))

20


In [3]:
print(books[0])

<article class="product_pod">
<div class="image_container">
<a href="catalogue/a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
</div>
<p class="star-rating Three">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
<div class="product_price">
<p class="price_color">£51.77</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>


###### The important concept: Website ->Request -> HTML ->BeautifulSoup -> Find repeating HTML element (article.product_pod) -> 20 individual products ->Extract:Title,Price,Rating,Availability,URL ->Pandas DataFrame -> CSV/Excel/SQL

##### Our goal is to turn that messy HTML into clean data. Use '"<a> to get the book title."

In [4]:
book = books[0]

title = book.h3.a["title"]

print(title)

A Light in the Attic


##### Extract the price

In [5]:
price = book.find("p", class_="price_color").text

print(price)

£51.77


##### Extract Availability

In [6]:
availability = book.find("p", class_="instock").text.strip()

print(availability)

In stock


###### Extract Rating

In [7]:
rating = book.find("p", class_="star-rating")["class"]

print(rating)

['star-rating', 'Three']


In [8]:
rating = book.find("p", class_="star-rating")["class"][1]

print(rating)

Three


###### Put it all together

In [9]:
book = books[0]

title = book.h3.a["title"]
price = book.find("p", class_="price_color").text
rating = book.find("p", class_="star-rating")["class"][1]
availability = book.find("p", class_="instock").text.strip()

print(title)
print(price)
print(rating)
print(availability)

A Light in the Attic
£51.77
Three
In stock


###### There is currency, we do not want that, we want it as a float.

In [10]:
price = book.find("p", class_="price_color").text
price = float(price.replace("£", ""))

print(price)

51.77


##### Do a loop through the 20 books in the page.

In [11]:
titles = []
prices = []
ratings = []
availability_list = []

for book in books:
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    price = float(price.replace("£", ""))
    rating = book.find("p", class_="star-rating")["class"][1]
    availability = book.find("p", class_="instock").text.strip()

    titles.append(title)
    prices.append(price)
    ratings.append(rating)
    availability_list.append(availability)

it looks like this
book 1 → extract values → add to lists
book 2 → extract values → add to lists
book 3 → extract values → add to lists
...
book 20 → extract values → add to lists

In [12]:
print(len(titles))
print(titles[:5])
print(prices[:5])

20
['A Light in the Attic', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History of Humankind']
[51.77, 53.74, 50.1, 47.82, 54.23]


###### Turn everything into a Pandas DataFrame

In [13]:
import pandas as pd

df = pd.DataFrame({
    "title": titles,
    "price": prices,
    "rating": ratings,
    "availability": availability_list
})

df.head()

,title,price,rating,availability
0,A Light in the Attic,51.77,Three,In stock
1,Tipping the Velvet,53.74,One,In stock
2,Soumission,50.10,One,In stock
3,Sharp Objects,47.82,Four,In stock
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         20 non-null     object 
 1   price         20 non-null     float64
 2   rating        20 non-null     object 
 3   availability  20 non-null     object 
dtypes: float64(1), object(3)
memory usage: 768.0+ bytes


##### URL Structure https://books.toscrape.com/catalogue/page-2.html

So there is a predictable pattern. 

In [15]:
for page in range(1, 51):
    print(page)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50


In [16]:
for page in range(1, 51):

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    print(url)

https://books.toscrape.com/catalogue/page-1.html
https://books.toscrape.com/catalogue/page-2.html
https://books.toscrape.com/catalogue/page-3.html
https://books.toscrape.com/catalogue/page-4.html
https://books.toscrape.com/catalogue/page-5.html
https://books.toscrape.com/catalogue/page-6.html
https://books.toscrape.com/catalogue/page-7.html
https://books.toscrape.com/catalogue/page-8.html
https://books.toscrape.com/catalogue/page-9.html
https://books.toscrape.com/catalogue/page-10.html
https://books.toscrape.com/catalogue/page-11.html
https://books.toscrape.com/catalogue/page-12.html
https://books.toscrape.com/catalogue/page-13.html
https://books.toscrape.com/catalogue/page-14.html
https://books.toscrape.com/catalogue/page-15.html
https://books.toscrape.com/catalogue/page-16.html
https://books.toscrape.com/catalogue/page-17.html
https://books.toscrape.com/catalogue/page-18.html
https://books.toscrape.com/catalogue/page-19.html
https://books.toscrape.com/catalogue/page-20.html
https://b

Create empty list first

In [17]:
all_titles = []
all_prices = []
all_ratings = []
all_availability = []

In [18]:
for page in range(1, 3):

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(url)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    print("Page:", page, "Books:", len(books))

Page: 1 Books: 20
Page: 2 Books: 20


We do a test for the first 2 pages and get page 1 and 2. Now we have 2 levels, we need to create a loop inside another loop. We use a nested loop.

In [19]:
for book in books:

    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    rating = book.find("p", class_="star-rating")["class"][1]
    availability = book.find("p", class_="instock").text.strip()

    all_titles.append(title)
    all_prices.append(price)
    all_ratings.append(rating)
    all_availability.append(availability)

In [20]:
print(len(all_titles))

20


In [21]:
import pandas as pd

df_all = pd.DataFrame({
    "title": all_titles,
    "price": all_prices,
    "rating": all_ratings,
    "availability": all_availability
})

In [22]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df_all["rating"] = df_all["rating"].map(rating_map)

In [23]:
df_all["price"] = (
    df_all["price"]
    .str.replace("£", "", regex=False)
    .astype(float)
)

In [24]:
df_all.head()

,title,price,rating,availability
0,In Her Wake,12.84,1,In stock
1,How Music Works,37.32,2,In stock
2,Foolproof Preserving: A Guide to Small Batch J...,30.52,3,In stock
3,Chase Me (Paris Nights #2),25.27,5,In stock
4,Black Dust,34.53,5,In stock


In [25]:
df_all.to_excel("books_scraped.xlsx", index=False)

In [26]:
import os
print(os.getcwd())

C:\Users\justi\OneDrive\Desktop\SMU\Python Self\Mining


Add product link 

In [27]:
relative_url = book.h3.a["href"]

product_url = "https://books.toscrape.com/catalogue/" + relative_url

In [28]:
print(product_url)

https://books.toscrape.com/catalogue/you-cant-bury-them-all-poems_961/index.html


Create a new storage list for the product links

In [29]:
all_urls = []

add URLs into the loop

In [30]:
all_urls.append(product_url)

In [31]:
print("titles:", len(all_titles))
print("prices:", len(all_prices))
print("ratings:", len(all_ratings))
print("availability:", len(all_availability))
print("urls:", len(all_urls))

titles: 20
prices: 20
ratings: 20
availability: 20
urls: 1


In [32]:
all_titles = []
all_prices = []
all_ratings = []
all_availability = []
all_urls = []

for page in range(1, 2):   # one page = 20 books for now

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(url)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:

        title = book.h3.a["title"]

        price = book.find("p", class_="price_color").text
        price = float(price.replace("£", ""))

        rating = book.find("p", class_="star-rating")["class"][1]

        availability = book.find(
            "p",
            class_="instock"
        ).text.strip()

        relative_url = book.h3.a["href"]

        product_url = (
            "https://books.toscrape.com/catalogue/"
            + relative_url
        )

        all_titles.append(title)
        all_prices.append(price)
        all_ratings.append(rating)
        all_availability.append(availability)
        all_urls.append(product_url)

In [33]:
df_all = pd.DataFrame({
    "title": all_titles,
    "price": all_prices,
    "rating": all_ratings,
    "availability": all_availability,
    "product_url": all_urls
})

In [34]:
print("titles:", len(all_titles))
print("prices:", len(all_prices))
print("ratings:", len(all_ratings))
print("availability:", len(all_availability))
print("urls:", len(all_urls))

titles: 20
prices: 20
ratings: 20
availability: 20
urls: 20


In [35]:
all_titles = []
all_prices = []
all_ratings = []
all_availability = []
all_urls = []

In [36]:
for page in range(1, 4):

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(url)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:

        title = book.h3.a["title"]

        price = book.find("p", class_="price_color").text
        price = float(price.replace("£", ""))

        rating = book.find("p", class_="star-rating")["class"][1]

        availability = book.find(
            "p",
            class_="instock"
        ).text.strip()

        relative_url = book.h3.a["href"]

        product_url = (
            "https://books.toscrape.com/catalogue/"
            + relative_url
        )

        all_titles.append(title)
        all_prices.append(price)
        all_ratings.append(rating)
        all_availability.append(availability)
        all_urls.append(product_url)

In [37]:
print(
    len(all_titles),
    len(all_prices),
    len(all_ratings),
    len(all_availability),
    len(all_urls)
)

60 60 60 60 60


In [38]:
df_all = pd.DataFrame({
    "title": all_titles,
    "price": all_prices,
    "rating": all_ratings,
    "availability": all_availability,
    "product_url": all_urls
})

Each product_url leads to a book page container the extra fields you want.

In [39]:
product_url = all_urls[0]

response = requests.get(product_url)
response.encoding = "utf-8"

product_soup = BeautifulSoup(response.text, "html.parser")

In [40]:
category = product_soup.select("ul.breadcrumb li a")[-1].text.strip()

print(category)

Poetry


The first book category that we get is Poetry

Next we get the UPC, tax, number of reviews, and stock quantity stored in the product information table.

In [41]:
table = product_soup.find("table", class_="table table-striped")

rows = table.find_all("tr")

for row in rows:
    print(row.th.text, ":", row.td.text)

UPC : a897fe39b1053632
Product Type : Books
Price (excl. tax) : £51.77
Price (incl. tax) : £51.77
Tax : £0.00
Availability : In stock (22 available)
Number of reviews : 0


In [42]:
product_data = {}

for row in rows:
    key = row.th.text.strip()
    value = row.td.text.strip()

    product_data[key] = value

In [43]:
print(product_data)

{'UPC': 'a897fe39b1053632', 'Product Type': 'Books', 'Price (excl. tax)': '£51.77', 'Price (incl. tax)': '£51.77', 'Tax': '£0.00', 'Availability': 'In stock (22 available)', 'Number of reviews': '0'}


In [44]:
upc = product_data["UPC"]

tax = product_data["Tax"]

reviews = product_data["Number of reviews"]

availability_detail = product_data["Availability"]

In [45]:
import re

quantity_match = re.search(r"\((\d+) available\)", availability_detail)

if quantity_match:
    availability_quantity = int(quantity_match.group(1))
else:
    availability_quantity = 0

In [46]:
print("Category:", category)
print("UPC:", upc)
print("Tax:", tax)
print("Reviews:", reviews)
print("Quantity:", availability_quantity)

Category: Poetry
UPC: a897fe39b1053632
Tax: £0.00
Reviews: 0
Quantity: 22


In [47]:
tax = float(tax.replace("£", ""))
reviews = int(reviews)

In [48]:
df_all.head()

,title,price,rating,availability,product_url
0,A Light in the Attic,51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...


In [49]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         60 non-null     object 
 1   price         60 non-null     float64
 2   rating        60 non-null     object 
 3   availability  60 non-null     object 
 4   product_url   60 non-null     object 
dtypes: float64(1), object(4)
memory usage: 2.5+ KB


In [50]:
df_all.isnull().sum()

title           0
price           0
rating          0
availability    0
product_url     0
dtype: int64

In [51]:
print("Total books:", len(df_all))

Total books: 60


In [52]:
df_all.to_csv(
    "books_scraped_detailed.csv",
    index=False
)

df_all.to_excel(
    "books_scraped_detailed.xlsx",
    index=False,
    engine="openpyxl"
)

Text Extraction on one book

In [53]:
import requests
from bs4 import BeautifulSoup

product_url = df_all["product_url"].iloc[0]

response = requests.get(product_url, timeout=15)
response.raise_for_status()
response.encoding = "utf-8"

product_soup = BeautifulSoup(response.text, "html.parser")

description_heading = product_soup.find("div", id="product_description")

if description_heading:
    description = description_heading.find_next_sibling("p").get_text(strip=True)
else:
    description = ""

print(description)

It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And your cradle, too?Baby, I think someone down here'sGot it in for you. Shel, you never sounded

###### Add descriptions to all my existing records

In [55]:
import pandas as pd

df_csv = pd.read_csv("books_scraped_detailed.csv")
df_xlsx = pd.read_excel("books_scraped_detailed.xlsx")

print("CSV columns:")
print(df_csv.columns.tolist())

print("\nExcel columns:")
print(df_xlsx.columns.tolist())

CSV columns:
['title', 'price', 'rating', 'availability', 'product_url']

Excel columns:
['title', 'price', 'rating', 'availability', 'product_url']


In [56]:
import requests
from bs4 import BeautifulSoup
import re
import time

all_categories = []
all_upc = []
all_tax = []
all_reviews = []
all_quantity = []

for i, product_url in enumerate(df_all["product_url"], start=1):

    try:
        response = requests.get(product_url, timeout=15)
        response.raise_for_status()
        response.encoding = "utf-8"

        soup = BeautifulSoup(response.text, "html.parser")

        # Category
        category = soup.select("ul.breadcrumb li a")[-1].text.strip()

        # Product information table
        table = soup.find("table", class_="table table-striped")

        product_data = {}

        for row in table.find_all("tr"):
            key = row.th.text.strip()
            value = row.td.text.strip()
            product_data[key] = value

        upc = product_data.get("UPC", "")
        tax = product_data.get("Tax", "£0.00")
        reviews = product_data.get("Number of reviews", "0")
        availability_detail = product_data.get("Availability", "")

        # Clean tax
        tax = float(tax.replace("£", ""))

        # Clean reviews
        reviews = int(reviews)

        # Extract quantity
        quantity_match = re.search(
            r"\((\d+) available\)",
            availability_detail
        )

        if quantity_match:
            quantity = int(quantity_match.group(1))
        else:
            quantity = 0

        all_categories.append(category)
        all_upc.append(upc)
        all_tax.append(tax)
        all_reviews.append(reviews)
        all_quantity.append(quantity)

    except Exception as e:
        print(f"Error on book {i}: {e}")

        all_categories.append("")
        all_upc.append("")
        all_tax.append(None)
        all_reviews.append(None)
        all_quantity.append(None)

    print(f"Processed {i}/{len(df_all)}")

    time.sleep(0.5)

Processed 1/60
Processed 2/60
Processed 3/60
Processed 4/60
Processed 5/60
Processed 6/60
Processed 7/60
Processed 8/60
Processed 9/60
Processed 10/60
Processed 11/60
Processed 12/60
Processed 13/60
Processed 14/60
Processed 15/60
Processed 16/60
Processed 17/60
Processed 18/60
Processed 19/60
Processed 20/60
Processed 21/60
Processed 22/60
Processed 23/60
Processed 24/60
Processed 25/60
Processed 26/60
Processed 27/60
Processed 28/60
Processed 29/60
Processed 30/60
Processed 31/60
Processed 32/60
Processed 33/60
Processed 34/60
Processed 35/60
Processed 36/60
Processed 37/60
Processed 38/60
Processed 39/60
Processed 40/60
Processed 41/60
Processed 42/60
Processed 43/60
Processed 44/60
Processed 45/60
Processed 46/60
Processed 47/60
Processed 48/60
Processed 49/60
Processed 50/60
Processed 51/60
Processed 52/60
Processed 53/60
Processed 54/60
Processed 55/60
Processed 56/60
Processed 57/60
Processed 58/60
Processed 59/60
Processed 60/60


In [57]:
print(len(df_all))
print(len(all_categories))
print(len(all_upc))
print(len(all_tax))
print(len(all_reviews))
print(len(all_quantity))

60
60
60
60
60
60


In [58]:
df_all["category"] = all_categories
df_all["UPC"] = all_upc
df_all["tax"] = all_tax
df_all["reviews"] = all_reviews
df_all["quantity"] = all_quantity

In [59]:
print(df_all.columns.tolist())

['title', 'price', 'rating', 'availability', 'product_url', 'category', 'UPC', 'tax', 'reviews', 'quantity']


In [61]:
print(len(df_all))
print(len(descriptions))

60
60


In [63]:
df_all["description"] = descriptions

In [64]:
df_all[
    ["title", "category", "description", "UPC", "quantity"]
].head()

,title,category,description,UPC,quantity
0,A Light in the Attic,Poetry,It's hard to imagine a world without A Light i...,a897fe39b1053632,22
1,Tipping the Velvet,Historical Fiction,"""Erotic and absorbing...Written with starling ...",90fa61229261140a,20
2,Soumission,Fiction,"Dans une France assez proche de la nôtre, un h...",6957f44c3847a760,20
3,Sharp Objects,Mystery,"WICKED above her hipbone, GIRL across her hear...",e00eb4fd7b871a48,20
4,Sapiens: A Brief History of Humankind,History,From a renowned historian comes a groundbreaki...,4165285e1663650f,20


In [65]:
df_all.to_csv(
    "books_scraped_detailed_v2.csv",
    index=False
)

df_all.to_excel(
    "books_scraped_detailed_v2.xlsx",
    index=False,
    engine="openpyxl"
)